# LSTM Time-Series Forecasting (PyTorch)

**Dataset:** [Sunspots](https://www.kaggle.com/datasets/robervalt/sunspots) — Kaggle
dataset `robervalt/sunspots`. Monthly mean total sunspot number from 1749 to 2017
(~3,265 monthly observations), sourced from SIDC (Solar Influences Data Analysis
Center, Royal Observatory of Belgium).

The dataset is downloaded **directly from Kaggle in code** below, using Kaggle's
official `kagglehub` library — no manual download or file upload needed.

> **One-time setup required:** Kaggle's API requires authentication even for public
> datasets. Before running this notebook:
> 1. Create a free Kaggle account if you don't have one.
> 2. Go to **kaggle.com → Settings → API → Create New Token**. This downloads a
>    `kaggle.json` file containing your username and API key.
> 3. Either:
>    - Place that file at `~/.kaggle/kaggle.json` (Linux/Mac) or
>      `C:\Users\<you>\.kaggle\kaggle.json` (Windows), **or**
>    - Set two environment variables before launching Jupyter:
>      `KAGGLE_USERNAME=<your_username>` and `KAGGLE_KEY=<your_api_key>`.
>
> Once that's set up, `kagglehub.dataset_download(...)` below will authenticate
> automatically and cache the files locally.

This notebook:

1. Downloads and loads the dataset directly from Kaggle
2. Scales the data and builds sliding-window sequences
3. Defines an LSTM regression model
4. Trains it with early stopping
5. Evaluates on a held-out test set (RMSE / MAE / MAPE)
6. Forecasts N future steps recursively (multi-step ahead)
7. Plots and saves the trained model to disk

In [ ]:
# One-time install if kagglehub isn't already available:
# !pip install kagglehub

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import kagglehub

%matplotlib inline

## 0. Reproducibility & device

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data — downloaded directly from Kaggle

`kagglehub.dataset_download()` fetches the dataset (requires the Kaggle API
credentials described above), caches it locally, and returns the local folder path
containing the CSV.

In [ ]:
# Downloads (or reuses the local cache of) the dataset directly from Kaggle
dataset_path = kagglehub.dataset_download("robervalt/sunspots")
print("Dataset downloaded to:", dataset_path)

import os
print("Files:", os.listdir(dataset_path))

In [ ]:
# The dataset ships as a single CSV with columns: Unnamed: 0, Date, Monthly Mean Total Sunspot Number
csv_path = os.path.join(dataset_path, os.listdir(dataset_path)[0])

raw = pd.read_csv(csv_path)
print(raw.shape)
raw.head()

In [ ]:
df = raw.rename(columns={
    "Date": "date",
    "Monthly Mean Total Sunspot Number": "value",
})[["date", "value"]].copy()

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["value"], linewidth=0.7)
plt.title("Monthly Sunspot Numbers (Kaggle: robervalt/sunspots)")
plt.xlabel("Date")
plt.ylabel("Sunspot count")
plt.grid(alpha=0.3)
plt.show()

## 2. Train / val / test split

Chronological split — **never shuffle time series data** across train/test.
The scaler is fit only on the training set to avoid leakage.

In [ ]:
values = df["value"].values.reshape(-1, 1)

train_frac, val_frac = 0.70, 0.15
n = len(values)
train_end = int(n * train_frac)
val_end = int(n * (train_frac + val_frac))

train_raw = values[:train_end]
val_raw = values[train_end:val_end]
test_raw = values[val_end:]

# Fit scaler ONLY on training data to avoid leakage
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print(f"Train size: {len(train_raw)}, Val size: {len(val_raw)}, Test size: {len(test_raw)}")

## 3. Sliding-window sequence creation

Sunspot cycles run ~9-14 years (108-168 months). A 24-month (2-year) look-back is
long enough to capture short-term momentum and the rising/falling phase of a cycle,
while keeping training fast. Try a longer `WINDOW` (e.g. 60 or 120) to see if the
model picks up more of the long cycle structure.

In [ ]:
def create_sequences(data: np.ndarray, window: int, horizon: int = 1):
    """Turn a (N,1) array into (X, y) sliding-window supervised pairs.
    X: (samples, window, 1)   y: (samples, horizon)
    """
    X, y = [], []
    for i in range(len(data) - window - horizon + 1):
        X.append(data[i: i + window])
        y.append(data[i + window: i + window + horizon, 0])
    return np.array(X), np.array(y)


WINDOW = 24      # look-back length (past 24 months)
HORIZON = 1      # forecast 1 step ahead (change for multi-step direct forecasting)

X_train, y_train = create_sequences(train_scaled, WINDOW, HORIZON)
X_val, y_val = create_sequences(val_scaled, WINDOW, HORIZON)
X_test, y_test = create_sequences(test_scaled, WINDOW, HORIZON)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 32
train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## 4. LSTM model

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, output_size),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)          # (batch, seq_len, hidden_size)
        last_step = lstm_out[:, -1, :]       # take the final time step's hidden state
        return self.fc(last_step)            # (batch, output_size)


model = LSTMForecaster(
    input_size=1, hidden_size=64, num_layers=2, output_size=HORIZON, dropout=0.2
).to(device)
print(model)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

## 5. Training loop with early stopping

In [ ]:
def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if train:
                optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

In [ ]:
EPOCHS = 100
PATIENCE = 10
best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": []}
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

model.load_state_dict(best_state)  # restore best weights

## 6. Evaluation on the test set (in original scale)

Note: sunspot counts include months with values near/at 0 (solar minimum), which makes
MAPE unstable (division by ~0) — we report it but lean more on RMSE/MAE here.

In [ ]:
model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    preds_scaled = model(X_test_t).cpu().numpy()

preds = scaler.inverse_transform(preds_scaled)
actuals = scaler.inverse_transform(y_test)

rmse = float(np.sqrt(np.mean((preds - actuals) ** 2)))
mae = float(np.mean(np.abs(preds - actuals)))

# Avoid div-by-zero blowups from near-zero sunspot months when computing MAPE
nonzero_mask = actuals.flatten() > 5
mape = float(np.mean(np.abs((actuals[nonzero_mask] - preds[nonzero_mask]) / actuals[nonzero_mask])) * 100)

print("--- Test set performance ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"MAPE (months with >5 sunspots): {mape:.2f}%")

## 7. Multi-step recursive forecast beyond the available data

In [ ]:
def recursive_forecast(model, last_window_scaled: np.ndarray, n_future: int) -> np.ndarray:
    """Feed the model's own predictions back in to forecast n_future steps ahead."""
    model.eval()
    window = last_window_scaled.copy()  # shape (WINDOW, 1)
    preds_scaled = []
    with torch.no_grad():
        for _ in range(n_future):
            x = torch.tensor(window[np.newaxis, :, :], dtype=torch.float32).to(device)
            next_val = model(x).cpu().numpy()[0, 0]
            preds_scaled.append(next_val)
            window = np.vstack([window[1:], [[next_val]]])
    return scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()


N_FUTURE = 60  # 5 years ahead
last_window = test_scaled[-WINDOW:]
future_preds = recursive_forecast(model, last_window, N_FUTURE)
future_dates = pd.date_range(df["date"].iloc[-1] + pd.DateOffset(months=1), periods=N_FUTURE, freq="MS")

print(f"First 5 future forecasts (sunspot count):\n{future_preds[:5]}")

## 8. Plots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14))

# Loss curves
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_title("Training / Validation Loss (MSE, scaled space)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Test predictions vs actuals
test_dates = df["date"].iloc[val_end + WINDOW: val_end + WINDOW + len(actuals)]
axes[1].plot(test_dates, actuals, label="Actual", linewidth=1.2)
axes[1].plot(test_dates, preds, label="Predicted", linewidth=1.2, linestyle="--")
axes[1].set_title(f"Test Set: Actual vs Predicted (RMSE={rmse:.2f}, MAE={mae:.2f})")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Sunspot count")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Full series + future forecast
axes[2].plot(df["date"], df["value"], label="Historical", linewidth=0.6)
axes[2].plot(future_dates, future_preds, label=f"Forecast (+{N_FUTURE} months)", linewidth=1.5, color="red")
axes[2].axvline(df["date"].iloc[-1], color="gray", linestyle=":", label="Forecast start")
axes[2].set_title("Full History + Future Forecast")
axes[2].set_xlabel("Date")
axes[2].set_ylabel("Sunspot count")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Save model + scaler for later reuse

You can reload this later with `torch.load(...)` and rebuild the `LSTMForecaster` using
the saved `model_config`.

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": {
            "input_size": 1, "hidden_size": 64, "num_layers": 2,
            "output_size": HORIZON, "dropout": 0.2, "window": WINDOW,
        },
        "scaler_min": scaler.min_,
        "scaler_scale": scaler.scale_,
    },
    "lstm_forecaster_sunspots.pt",
)
print("Saved model to lstm_forecaster_sunspots.pt")